In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
from T_method import LayeredStructure
from My_plotter import Plotter, Style
from Global_optimizer import my_json_load 

In [ ]:
a = [1, 2, 3, 4, 5]

In [ ]:
MU_0 = 4e-7 * np.pi
EPS_0 = 8.8541878188e-12
ETA_0 = np.sqrt(MU_0/EPS_0)
C = 1/np.sqrt(MU_0*EPS_0)

In [ ]:
a = 8e-3
f_min = 3.3e9
f_max = 4.2e9
f_0 = (f_min + f_max)/2
print(f'центральная частота - {(f_0*1.e-9):.2f} ГГц')
lamb_0 = C/f_0
k_0 = 2*np.pi/lamb_0
print(f'длина волны на центральной частоте - {(lamb_0*1000):.2f} мм')
print(f'волновое число на центральной частоте - {(k_0):.2f} 1/м')
delta = a*3

In [ ]:
st = Style()
fig, ax = plt.subplots()
pl = Plotter(ax, st)
df = np.linspace(-50, 50, 400)/100
alpha = np.array([1.5, 1.5, 1.5])
structure = LayeredStructure(alpha, beta_d=np.pi)
directivity = 10*np.log10(structure.directivity(df))
directivity_two_sources = 10*np.log10(structure.directivity_two_sources_diagonal(df))
pl.plot(df, directivity, label='Directivity')
pl.plot(df, directivity_two_sources, label='Directivity (Two Sources)', linestyle='--')
pl.finalize()
ax.axhline(19, color='gray', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
theta = np.array([1,2])
df = np.array([0, 0.1, 0.2])
res = theta[None, :, None] + df[None, None, :]
print(np.shape(res))
print(np.shape(res[:, None]))

In [ ]:
import scipy as sc
from scipy.special import j0, j1
y = np.linspace(0, 1, 200)
def f(y, real=True):
    if real:
        return lambda x: np.cos(x)**2*np.cos(y*(np.sin(x)+np.cos(x)))
    return lambda x: np.cos(x)**2*np.sin(y*(np.sin(x)+np.cos(x)))

resf = np.zeros_like(y, dtype=np.complex128)

for yi in range(len(y)):
    resf[yi] = sc.integrate.quad(f(y[yi], real=True), 0, np.pi*2)[0] + 1.j*sc.integrate.quad(f(y[yi], real=False), 0, np.pi*2)[0]

def integral_result(y):
    # Обработка y = 0 (предел)
    if np.isscalar(y) and y == 0:
        return np.pi / 2  # предел при y->0
    # Общий случай
    sqrt2_y = np.sqrt(2) * y
    return np.pi * (j0(sqrt2_y))

res_analytical = np.array([integral_result(yi) for yi in y])

plt.plot(y, resf, label='Numerical Integration')
plt.plot(y, res_analytical, label='Analytical Result', linestyle='--')
plt.xlabel('y')
plt.ylabel('Integral Result')
plt.legend()
plt.title('Comparison of Numerical and Analytical Integration Results')
plt.show()


In [ ]:
from scipy.special import jv
# Evaluate the 0th-order Bessel function at x=1
print(jv(0, 0)) 

In [ ]:
st = Style()
fig, ax = plt.subplots()
pl1 = Plotter(ax, st)
df = np.linspace(-30, 30, 200)/100
for seed in [90]:
    path = Path(r"results\final\cl10_wide.json")
    data = my_json_load(path)
    alpha = data["best_alpha"]
    print(alpha)
    beta = data["best_beta"]
    alpha_l = data["best_alpha_l"]
    alpha_c = data["best_alpha_c"]
    dipole_shift = data["best_dipole_shift"]
    beta_d = data["best_beta_d"]
    structure = LayeredStructure(alpha, beta=beta, alpha_l=alpha_l, alpha_c=alpha_c, dipole_shift=dipole_shift, beta_d=beta_d)
    directivity = 10*np.log10(structure.directivity_two_sources_diagonal(df))
    pl1.plot((1+df)*f_0*1.e-9, directivity, label=f"18dBi")
pl1.finalize()
pl1.set_ylim((0, 25))
ax.axhline(17.5, color='gray', linestyle='--', alpha=0.5)
ax.axhline(16, color='gray', linestyle='--', alpha=0.5)
ax.axvline(f_max*1.e-9, color='gray', linestyle='--', alpha=0.5)
ax.axvline(f_min*1.e-9, color='gray', linestyle='--', alpha=0.5)
# ax.axvline((1-0.15)*f_0*1.e-9, color='orange', linestyle='--', alpha=0.5)
# ax.axvline((1+0.15)*f_0*1.e-9, color='orange', linestyle='--', alpha=0.5)
ax.axvline((1)*f_0*1.e-9, color='orange', linestyle='--', alpha=0.5)
ax.axvline
ax.set_xlabel('Frequency (GHz)')
ax.set_ylabel('Directivity (dBi)')
ax.minorticks_on()
plt.show()

In [ ]:
from MyRadiationPattern import RadiationPattern3DPlotter
f = np.array([3.3, 3.5, 3.7, 3.9, 4.2])*1.e9
df = (f - f_0)/f_0
phi_arr = np.linspace(0, 2*np.pi, 200)
theta_arr = np.linspace(0, np.pi/2, 200)
for i, f_i in enumerate(df):
    dir = structure.radiation_pattern_two_sources_diagonal(phi_arr, theta_arr, np.array([f_i]), mode='absolute')[0, :, :]
    rppl = RadiationPattern3DPlotter()
    rppl.set_title(f'Frequency={f[i]*1.e-9} GHz')
    rppl.plot(phi_arr*180/np.pi, theta_arr*180/np.pi, dir, mode='logarithmic', threshold=-15)

In [ ]:
# import Global_optimizer as glo
# optp = glo.OptimizationParameters()
# optp.penalty_c = 0.0
# optp.mode = 'work'
# optp.n = 2
# optp.bounds_alpha = [(-20.0, 0.0), (-20.0, 0.0)]
# optp.bounds_beta = [(0.2, 7.0), (0.2, 1.0)]
# optp.bounds_alpha_screen = [(0, 20.0), (-20, 0.0)]
# optp.bounds_dipole_shift = [(0.3, np.pi)]
# optp.bounds_beta_d = [(0.0, 9.0)]
# optp.eps = 1e-3
# optp.limit = 200
# optp.sigma_thres = 16 - 0.5
# optp.segment_width_percent = 30
# optp.segment_shift_percent = 0
# print(optp.bounds)
# print(len(optp.df_center_segment))
# params = np.array([alpha[0], alpha[1], beta[0], beta[1], alpha_l, alpha_c, dipole_shift, beta_d])
# print(params)
# k = optp.k
# b = optp.b
# normalized_params = (params - b) / k
# print(normalized_params)
# print(optp.objective_function_two_sources(normalized_params))

In [ ]:
# from scipy.integrate import dblquad
# st = Style()
# fig, ax = plt.subplots()
# pl = Plotter(ax, st)
# df_dir = np.linspace(-30, 30, 20)/100
# def integrand(df):
#     df = np.array([df])
#     def temp_integrand(theta, phi):
#         phi = np.array([phi])
#         theta = np.array([theta])
#         f = structure.radiation_pattern_two_sources_diagonal( phi, theta, df)
#         return f[0,0,0]*np.sin(theta[0])
#     return temp_integrand
# p_normal = structure.radiation_pattern_two_sources_diagonal(np.array([0]), np.array([0]), df_dir)[:,0,0]
# directivity_direct = np.zeros_like(df_dir)
# for i, df_val in enumerate(df_dir):
#     directivity_direct[i] = p_normal[i]*4*np.pi/dblquad(integrand(df_val), 0, 2*np.pi, 0, np.pi/2, epsabs=1e-3, epsrel=1e-3)[0]
#     print(f"Directivity at df={df_val}: {directivity_direct[i]}")
# pl.plot((1+df)*f_0*1.e-9, directivity, label=f"18dBi")
# pl.plot((1+df_dir)*f_0*1.e-9, 10*np.log10(directivity_direct), label='Directivity (Direct Integration)', linestyle='--')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
from T_method import LayeredStructure
from My_plotter import Plotter, Style
from Global_optimizer import my_json_load 

st = Style()
fig, ax = plt.subplots()
pl = Plotter(ax, st)
f = np.array([3.3, 3.5, 3.7, 3.9, 4.2])*1.e9
df = (f - f_0)/f_0
phi = np.array([np.pi/4])
theor_theta = np.linspace(-np.pi/2, np.pi/2, 200)
path = Path(r"results\final\cc10.json")
data = my_json_load(path)
alpha = data["best_alpha"]
beta = data["best_beta"]
alpha_l = data["best_alpha_l"]
alpha_c = data["best_alpha_c"]
dipole_shift = data["best_dipole_shift"]
beta_d = data["best_beta_d"]

structure = LayeredStructure(alpha, beta=beta, alpha_l=alpha_l, alpha_c=alpha_c, dipole_shift=dipole_shift, beta_d=beta_d)
directivity = 10*np.log10(structure.directivity_two_sources_diagonal(df))
print("directivity=",  directivity)
pl.set_p(f*1.e-9)
for i, f_i in enumerate(f):
    radiation_pattern = structure.radiation_pattern_two_sources_diagonal(phi, theor_theta, np.array([df[i]]), mode='absolute')[0, 0, :]
    pl.multiple_plot(theor_theta*180/np.pi, radiation_pattern, f_i*1.e-9, label=f"f={f_i*1.e-9:.2f} GHz")
pl.set_ylim((-5,100))
pl.set_xlabel("Theta (deg)")
pl.set_ylabel("Radiation Pattern (dB)")
pl.set_title("Horizontal-plane RP for Different Frequencies")
pl.finalize()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
from T_method import LayeredStructure
from My_plotter import Plotter, Style
from Global_optimizer import my_json_load 

st = Style()
fig, ax = plt.subplots()
pl = Plotter(ax, st)
df = np.array([0])
phi = np.array([np.pi/2])
theor_theta = np.linspace(-np.pi/2, np.pi/2, 200)
path = Path(r"results\final\cc"+f"90.json")
data = my_json_load(path)
alpha = [ETA_0/-44.272, ETA_0/-99.4]
beta = [k_0*36.08340459e-3, k_0*3.82406191e-3]
alpha_l = ETA_0/100.1
alpha_c = -ETA_0/1000000000
dipole_shift = 10.5084e-3*k_0
beta_d = data["best_beta_d"]

structure = LayeredStructure(alpha, beta=beta, alpha_l=alpha_l, alpha_c=alpha_c, dipole_shift=dipole_shift, beta_d=0)
directivity = 10*np.log10(structure.directivity_two_sources_diagonal(df))
print("directivity=",  directivity)
radiation_pattern = structure.radiation_pattern_two_sources_diagonal(phi, theor_theta, df)[0, 0, :]
print(radiation_pattern[100:105])
pl.plot(theor_theta, radiation_pattern)

In [ ]:
print(f'best_alpha_Z = {ETA_0/data["best_alpha"]}')
print(f'best_beta_l = {data["best_beta"]/k_0*1000}')
print(f'best_alpha_l_Z = {ETA_0/data["best_alpha_l"]}')
print(f'best_alpha_c_Z = {ETA_0/data["best_alpha_c"]}')
print(f'best_dipole_shift_l = {data["best_dipole_shift"]/k_0*1000}')
print(f'best_beta_d_l = {data["best_beta_d"]/k_0*1000}')


In [ ]:
def sym_array(arr):
    return np.concatenate((-arr[:0:-1], arr[0:]))
path_1 = Path(r"C:\Users\Michael\Desktop\Project_X\cst_verification\res_RP\second_sample")
theta = np.arange(0, 89, 2, dtype=int)
s = 8e-3**2
e_f0 = []
for i in range(len(theta)):
    data = np.loadtxt(path_1 / f"theta={theta[i]}.txt", delimiter=',')
    e_f0.append(data[1, 1] + 1.j*data[1, 2])
e_f0 = np.array(e_f0)
e_f0_xip = np.abs(e_f0)**2/(ETA_0/(s*np.cos(np.radians(theta))))
st = Style()
fig, ax = plt.subplots()
pl = Plotter(ax, st)
theta = sym_array(theta)
e_f0_xip = np.abs(sym_array(e_f0_xip))
pl.plot(theta, np.abs(e_f0_xip), label='CST Simulation')
pl.plot(theor_theta*180/np.pi, radiation_pattern*2, label='T-Method Prediction', linestyle='--')


theory_theta = theor_theta * 180 / np.pi
theory_pattern = radiation_pattern * 2

# относительная ошибка (%)
theory_interp = np.interp(theta, theory_theta, theory_pattern)
rel_error = np.abs(e_f0_xip - theory_interp) / theory_interp * 100
ax2 = ax.twinx()
ax2.plot(theta, rel_error, color="red", linewidth=1.5, label="Relative Error", linestyle='--')
ax2.set_ylabel("Relative Error (%)", color="red")
ax2.tick_params(axis="y", colors="red")
pl.set_xlabel("Theta (deg)")
pl.set_ylabel("Radiation Pattern (linear)")
pl.finalize()
plt.show()

In [ ]:
print(10*np.log10(4/((np.sin(np.radians(15))**2))))

In [ ]:
print(np.arange(0, 89, 2, dtype=int))

In [ ]:
import csv
f_cst = np.linspace(3.0, 4.5, 200)
x_1 = ETA_0/(data["best_alpha"])[0]*(f_0/f_cst/1.e9)
x_2 = ETA_0/(data["best_alpha"])[1]*(f_0/f_cst/1.e9)
weights = np.ones_like(f_cst)
r = np.zeros_like(f_cst)
data_z1 = np.array([f_cst, r, x_1, weights]).T
path = Path(r"dispersion_for_cst\z_1.txt")
path.parent.mkdir(parents=True, exist_ok=True)
with open(path, 'w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file, delimiter='\t')
    writer.writerows(data_z1.tolist())
data_z2 = np.array([f_cst, r, x_2, weights]).T
path = Path(r"dispersion_for_cst\z_2.txt")
path.parent.mkdir(parents=True, exist_ok=True)
with open(path, 'w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file, delimiter='\t')
    writer.writerows(data_z2.tolist())

In [ ]:
f_cst = np.linspace(3.0, 4.5, 200)
y_l = data["best_alpha_l"]/ETA_0
y_c = data["best_alpha_c"]/ETA_0
f_rel = f_cst/f_0*1.e9
x_scr = 1/(y_l/f_rel+ y_c*f_rel)
weights = np.ones_like(f_cst)
r = np.zeros_like(f_cst)
data_z1 = np.array([f_cst, r, x_scr, weights]).T
path = Path(r"dispersion_for_cst\z_scr.txt")
path.parent.mkdir(parents=True, exist_ok=True)
with open(path, 'w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file, delimiter='\t')
    writer.writerows(data_z1.tolist())